# Theme Park Daily Attendance — XGBoost with Regional Flight-Volume Features

A remake of the ZooTampa attendance-projection notebook on a fully public, reproducible
stack, since the original CSVs are gone.

**Attendance & weather:** the Kaggle dataset
[`ayushtankha/hackathon`](https://www.kaggle.com/datasets/ayushtankha/hackathon)
(listed as "Disneyland Visitors Data" — the facilities inside are actually
**PortAventura World** (Salou, Spain) and **Tivoli Gardens** (Copenhagen), plus
15-min ride wait times, schedules, and hourly weather).

**Flight volumes:** because these parks are European, US BTS data doesn't apply.
Instead we use **EUROCONTROL Aviation Intelligence Unit — Airport Traffic**: daily IFR
arrivals and departures for every European airport, Jan 2016 – present, free per-year
CSVs (~7 MB each, no key). Park → airport mapping:

| Park | Airports (ICAO) |
|---|---|
| PortAventura World | LEBL (Barcelona, ~1 h) + LERS (Reus, ~15 min) |
| Tivoli Gardens | EKCH (Copenhagen) |

The pipeline mirrors the original: daily target (`Entries`), weather + calendar +
holiday features, `log1p` target, attendance lags, then the **18 flight features**
(last / current / next × day / week / month totals for arrivals *and* departures),
with a with/without impact comparison, feature importances, and a chronological
hold-out.

## Setup

In [ ]:
%pip install -q kagglehub holidays xgboost


In [ ]:
# Download the Kaggle dataset (public — no credentials usually needed;
# if you hit a 403, run kagglehub.login() or set KAGGLE_USERNAME / KAGGLE_KEY)
import kagglehub

path = kagglehub.dataset_download("ayushtankha/hackathon")
print("Path to dataset files:", path)


In [ ]:
# Inspect what we actually received — file names on Kaggle datasets can drift,
# so every loader below discovers its file/columns rather than hard-coding them.
from pathlib import Path
import pandas as pd

DATA_DIR = Path(path)
files = sorted(p for p in DATA_DIR.rglob("*") if p.is_file())
for p in files:
    print(f"{p.relative_to(DATA_DIR)}  ({p.stat().st_size/1e6:.1f} MB)")
    if p.suffix.lower() in (".csv", ".tsv"):
        try:
            peek = pd.read_csv(p, nrows=3, sep=None, engine="python")
            print("   columns:", list(peek.columns))
        except Exception as e:
            print("   (could not preview:", e, ")")


## Configuration

In [ ]:
import numpy as np

PARK = "PortAventura World"          # or "Tivoli Gardens"

PARK_AIRPORTS = {
    "PortAventura World": ["LEBL", "LERS"],   # Barcelona El Prat + Reus
    "Tivoli Gardens":     ["EKCH"],           # Copenhagen Kastrup
}
PARK_HOLIDAYS = {                     # (country, subdivision) for the holidays lib
    "PortAventura World": ("ES", "CT"),       # Spain, Catalonia
    "Tivoli Gardens":     ("DK", None),       # Denmark
}

AIRPORTS = PARK_AIRPORTS[PARK]

# The COVID collapse (spring 2020 onward) hits attendance AND flights together,
# which can make flight features look artificially strong. Flip this on to test
# how the model behaves on non-COVID years only.
DROP_COVID = False
COVID_START, COVID_END = "2020-03-01", "2021-06-30"

CACHE_DIR = Path("/content/flight_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print(f"Park: {PARK} | Airports: {AIRPORTS}")


## Attendance (target)
Daily attendance per facility. Non-positive values (closure days / data quirks —
notably the COVID period) are dropped with a report. Note this park operates
**seasonally**, so the date index has gaps: row-based lags later mean
"previous *operating* day", which is also what a demand planner would use.

In [ ]:
att_file = next((p for p in files if "attendance" in p.name.lower()), None)
assert att_file is not None, f"No attendance file found in {DATA_DIR}"

att_raw = pd.read_csv(att_file, sep=None, engine="python")
att_raw.columns = [c.strip() for c in att_raw.columns]
low = {c.lower(): c for c in att_raw.columns}

date_col = next((low[c] for c in ("usage_date", "date", "work_date", "day") if c in low), None)
park_col = next((low[c] for c in ("facility_name", "park", "facility", "site") if c in low), None)
target_col = next((low[c] for c in ("attendance", "entries", "visitors", "guests") if c in low), None)
assert date_col and target_col, (
    f"Could not identify date/target columns in {att_file.name}: {list(att_raw.columns)}")

print(f"Using columns: date={date_col!r}, park={park_col!r}, target={target_col!r}")
if park_col:
    print("Facilities present:", att_raw[park_col].unique())

df = att_raw.copy()
if park_col:
    df = df[df[park_col] == PARK]
df = df[[date_col, target_col]].rename(columns={date_col: "Date", target_col: "Entries"})
df["Date"] = pd.to_datetime(df["Date"])
df["Entries"] = pd.to_numeric(df["Entries"], errors="coerce")

n0 = len(df)
df = df.dropna(subset=["Entries"])
df = df[df["Entries"] > 0]
print(f"Dropped {n0 - len(df)} rows with missing/non-positive attendance")

if DROP_COVID:
    n1 = len(df)
    df = df[(df["Date"] < COVID_START) | (df["Date"] > COVID_END)]
    print(f"DROP_COVID: removed {n1 - len(df)} rows in [{COVID_START}, {COVID_END}]")

df = (df.groupby("Date", as_index=False)["Entries"].sum()
        .sort_values("Date").reset_index(drop=True))
print(f"{len(df)} operating days: {df['Date'].min():%Y-%m-%d} -> {df['Date'].max():%Y-%m-%d}")

import matplotlib.pyplot as plt
plt.figure(figsize=(14, 4))
plt.plot(df["Date"], df["Entries"], lw=0.8)
plt.title(f"{PARK} — daily attendance")
plt.tight_layout(); plt.show()


## Weather features
The dataset ships hourly weather (OpenWeather-style export). We aggregate to daily:
mean/max/min temperature, mean humidity, total rain, mean wind and cloud cover —
the same signal set as the original notebook's weather columns. Hours are UTC while
the park runs on local time; at daily grain the boundary shift is negligible.
If no weather file or usable columns are found, the pipeline continues without
weather rather than failing.

In [ ]:
wx_file = next((p for p in files if "weather" in p.name.lower() and p.suffix.lower() == ".csv"), None)
WEATHER_COLS = []

if wx_file is None:
    print("No weather file found — continuing without weather features.")
else:
    wx = pd.read_csv(wx_file, sep=None, engine="python")
    wx.columns = [c.strip() for c in wx.columns]
    wlow = {c.lower(): c for c in wx.columns}

    ts_col = next((wlow[c] for c in ("dt_iso", "datetime", "date_time", "date", "dt") if c in wlow), None)
    assert ts_col is not None, f"No timestamp column recognized in {wx_file.name}: {list(wx.columns)}"

    ts = pd.to_datetime(wx[ts_col], errors="coerce", utc=True)
    if ts.isna().mean() > 0.5:  # e.g. "2018-06-01 00:00:00 +0000 UTC" strings
        ts = pd.to_datetime(wx[ts_col].astype(str).str.slice(0, 19), errors="coerce", utc=True)
    wx["Date"] = ts.dt.tz_localize(None).dt.normalize()

    agg_spec = {  # source-candidates -> (output name, how)
        "temp":       ("Avg_Temp_C",       "mean"),
        "temp_max":   ("Max_Temp_C",       "max"),
        "temp_min":   ("Min_Temp_C",       "min"),
        "humidity":   ("Avg_Humidity_pct", "mean"),
        "rain_1h":    ("Total_Rain_mm",    "sum"),
        "wind_speed": ("Avg_Wind_ms",      "mean"),
        "clouds_all": ("Avg_Clouds_pct",   "mean"),
    }
    plan = {}
    for src, (out, how) in agg_spec.items():
        if src in wlow:
            col = wlow[src]
            wx[col] = pd.to_numeric(wx[col], errors="coerce")
            if how == "sum":
                wx[col] = wx[col].fillna(0)
            plan[col] = (out, how)

    daily_wx = wx.groupby("Date").agg({c: how for c, (out, how) in plan.items()})
    daily_wx.columns = [plan[c][0] for c in daily_wx.columns]
    daily_wx = daily_wx.round(2).reset_index()
    WEATHER_COLS = [c for c in daily_wx.columns if c != "Date"]

    df = pd.merge(df, daily_wx, on="Date", how="left")
    print(f"Weather features from {wx_file.name}: {WEATHER_COLS}")
    print(f"Attendance days with weather: {df[WEATHER_COLS].notna().all(axis=1).mean():.1%}")
display(df.head())


## Calendar & holiday features
Same treatment as the original: day-of-week, weekend flag, month/day, cyclical
day-of-year (sin/cos), plus public holidays for the park's country — including the
regional subdivision (Catalonia for PortAventura), which national-only calendars miss.

In [ ]:
import holidays as holidays_lib

country, subdiv = PARK_HOLIDAYS[PARK]
hol = holidays_lib.country_holidays(country, subdiv=subdiv,
                                    years=range(df["Date"].dt.year.min(),
                                                df["Date"].dt.year.max() + 2))

df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Is_Weekend"] = df["DayOfWeek"].isin([5, 6]).astype(int)
df["Month"] = df["Date"].dt.month
df["DayOfMonth"] = df["Date"].dt.day
df["Is_Holiday"] = df["Date"].apply(lambda d: int(d in hol))

doy = df["Date"].dt.dayofyear
df["DayOfYear_Sin"] = np.sin(2 * np.pi * doy / 365.25)
df["DayOfYear_Cos"] = np.cos(2 * np.pi * doy / 365.25)

print(f"Holiday calendar: {country}" + (f" / {subdiv}" if subdiv else ""),
      f"- {df['Is_Holiday'].sum()} holiday operating days")
display(df.head())


## Regional Flight Volume Features — EUROCONTROL Airport Traffic

**Source.** [EUROCONTROL AIU "Airport traffic"](https://ansperformance.eu/data/):
daily **IFR arrivals** (`FLT_ARR_1`) and **departures** (`FLT_DEP_1`) per airport
(`APT_ICAO`), Jan 2016 – present, published as per-year CSVs (~7 MB) at
`https://www.eurocontrol.int/performance/data/download/csv/airport_traffic_<YYYY>.csv`
— free, keyless, and it fully covers this dataset's 2018–2022 span with buffer on
both sides. Counts are summed across the park's airports.

**Window definitions** (for Arrivals `Arr_*` and Departures `Dep_*`, *t* = row's date):

| Feature | Window |
|---|---|
| `*_Last_Day` / `*_Curr_Day` / `*_Next_Day` | value on *t−1* / *t* / *t+1* |
| `*_Last_Week` | total over [*t−7*, *t−1*] |
| `*_Curr_Week` | total over the ISO week (Mon–Sun) containing *t* |
| `*_Next_Week` | total over [*t+1*, *t+7*] |
| `*_Last_Month` | total over [*t−30*, *t−1*] |
| `*_Curr_Month` | total over the calendar month containing *t* |
| `*_Next_Month` | total over [*t+1*, *t+30*] |

**Caveats worth knowing:**
- IFR movements count *all* instrument flights (passenger, cargo, some GA) — a
  consistent proxy for visitor inflow, not a passenger count.
- These are **actuals**, not schedules. `Curr_*`/`Next_*` are legitimate prediction-time
  features because airline schedules are published months ahead; training on actuals is
  the standard proxy (actual ≈ scheduled − cancellations). In deployment you'd source
  the forward windows from published schedules.
- Flights and attendance **both collapsed during COVID**, so flight features partly
  encode "is the world open" — impressive importances across 2020–21 deserve
  skepticism. Use `DROP_COVID = True` above to sanity-check on normal years.

In [ ]:
import requests

EUROCONTROL_CSV = ("https://www.eurocontrol.int/performance/data/download/csv/"
                   "airport_traffic_{year}.csv")

def fetch_airport_traffic_year(year, session=None, retries=3):
    """Download (and cache) one year of EUROCONTROL airport traffic. Returns a
    DataFrame or None if that year isn't published."""
    raw_file = CACHE_DIR / f"airport_traffic_{year}.csv"
    if not raw_file.exists():
        session = session or requests.Session()
        url = EUROCONTROL_CSV.format(year=year)
        last_err = None
        for attempt in range(retries):
            try:
                resp = session.get(url, timeout=180)
                if resp.status_code == 404:
                    return None
                resp.raise_for_status()
                raw_file.write_bytes(resp.content)
                break
            except requests.exceptions.RequestException as e:
                last_err = e
                import time as _t; _t.sleep(5 * (attempt + 1))
        else:
            raise RuntimeError(f"Could not download {url}: {last_err}")
    return pd.read_csv(raw_file)

def daily_airport_counts(years, airports):
    """Daily Arrivals/Departures summed over `airports` for the given years."""
    frames = []
    sess = requests.Session()
    for y in years:
        print(f"EUROCONTROL {y} ...", end=" ")
        raw = fetch_airport_traffic_year(y, session=sess)
        if raw is None:
            print("not published"); continue
        raw.columns = [c.strip().upper() for c in raw.columns]
        part = raw[raw["APT_ICAO"].isin(airports)].copy()
        part["Date"] = pd.to_datetime(part["FLT_DATE"].astype(str).str.strip(),
                                      errors="coerce")
        part = (part.groupby("Date")[["FLT_ARR_1", "FLT_DEP_1"]].sum()
                    .rename(columns={"FLT_ARR_1": "Arrivals", "FLT_DEP_1": "Departures"}))
        frames.append(part)
        print(f"{len(part)} days")
    out = pd.concat(frames).sort_index()
    return out.reset_index()

# Cover the attendance range plus 35-day buffers (Next_Month needs +30 real days)
from datetime import timedelta
_start = df["Date"].min() - timedelta(days=35)
_end = df["Date"].max() + timedelta(days=35)

flight_daily = daily_airport_counts(range(_start.year, _end.year + 1), AIRPORTS)
flight_daily = flight_daily[(flight_daily["Date"] >= _start) & (flight_daily["Date"] <= _end)]
flight_daily = flight_daily.reset_index(drop=True)

print(f"\nFlight coverage: {flight_daily['Date'].min():%Y-%m-%d} -> "
      f"{flight_daily['Date'].max():%Y-%m-%d} ({len(flight_daily)} days, "
      f"airports={AIRPORTS})")
display(flight_daily.describe().loc[["mean", "min", "max"]].round(1))


In [ ]:
def build_flight_window_features(flight_daily, prefix_map=(("Arrivals", "Arr"), ("Departures", "Dep"))):
    """Build last / current / next  x  day / week / month totals from a daily
    table with columns ['Date', 'Arrivals', 'Departures'].

    Leading windows use the identity: sum over [t+1, t+w] equals the trailing
    w-day rolling sum evaluated at t+w, shifted back to t.
    """
    fd = flight_daily.copy()
    fd["Date"] = pd.to_datetime(fd["Date"])
    fd = fd.sort_values("Date").set_index("Date")

    # Continuous daily index so windows align with calendar days; a missing day
    # becomes NaN and correctly poisons every window that touches it.
    fd = fd.reindex(pd.date_range(fd.index.min(), fd.index.max(), freq="D"))
    fd.index.name = "Date"

    out = pd.DataFrame(index=fd.index)
    iso = fd.index.isocalendar()
    week_key = iso["year"].astype(str) + "-W" + iso["week"].astype(str)
    month_key = fd.index.to_period("M")

    for col, pfx in prefix_map:
        s = fd[col].astype(float)
        out[f"{pfx}_Last_Day"] = s.shift(1)
        out[f"{pfx}_Curr_Day"] = s
        out[f"{pfx}_Next_Day"] = s.shift(-1)

        out[f"{pfx}_Last_Week"] = s.shift(1).rolling(7, min_periods=7).sum()
        out[f"{pfx}_Curr_Week"] = s.groupby(week_key.values).transform("sum")
        out[f"{pfx}_Next_Week"] = s.rolling(7, min_periods=7).sum().shift(-7)

        out[f"{pfx}_Last_Month"] = s.shift(1).rolling(30, min_periods=30).sum()
        out[f"{pfx}_Curr_Month"] = s.groupby(month_key).transform("sum")
        out[f"{pfx}_Next_Month"] = s.rolling(30, min_periods=30).sum().shift(-30)

    return out.reset_index()


flight_features = build_flight_window_features(flight_daily)
FLIGHT_FEATURE_COLS = [c for c in flight_features.columns if c != "Date"]
print(f"{len(FLIGHT_FEATURE_COLS)} flight features built:")
print(FLIGHT_FEATURE_COLS)
display(flight_features.dropna().head())


In [ ]:
# Merge into the working dataframe (drop-first so this cell is safe to re-run)
df = df.drop(columns=[c for c in FLIGHT_FEATURE_COLS if c in df.columns])
df = pd.merge(df, flight_features, on="Date", how="left")

_complete = df[FLIGHT_FEATURE_COLS].notna().all(axis=1).mean()
print(f"Operating days with complete flight features: {_complete:.1%}")
if _complete < 1:
    print("Edge rows near the limits of EUROCONTROL coverage carry NaNs; "
          "XGBoost handles NaN natively, but the dropna() after lag creation "
          "below will remove them from training.")
display(df[["Date", "Entries"] + FLIGHT_FEATURE_COLS[:4]].tail(3))


## Attendance lags
`Entries_Lag_1`, `Entries_Lag_7`, and a leakage-safe 7-row rolling mean, as in the
original. Because the park is seasonal, these are lags over *operating days*
(the previous open day / the open day 7 rows back), not strict calendar days.

In [ ]:
df = df.sort_values("Date").reset_index(drop=True)
df["Entries_Lag_1"] = df["Entries"].shift(1)
df["Entries_Lag_7"] = df["Entries"].shift(7)
df["Entries_Roll_Mean_7"] = df["Entries"].shift(1).rolling(7).mean()

n0 = len(df)
df = df.dropna().reset_index(drop=True)
print(f"Dropped {n0 - len(df)} edge rows with NaNs (lags / flight windows); {len(df)} remain")

# Numeric date key for trend, then Entries last
df["Date_Num"] = df["Date"].dt.strftime("%Y%m%d").astype(int)
model_df = df.drop(columns=["Date"])
model_df["Entries"] = model_df.pop("Entries")
display(model_df.head())


## Model — tuned XGBoost (random split)
Same recipe as the original's best pipeline: `log1p` target, 15% random test split,
regularized XGBoost. Random splits let the model interpolate with adjacent-day
features; the chronological split further down is the honest forecasting view.

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X_all = model_df.drop(columns=["Entries"])
y_all = np.log1p(model_df["Entries"].values)

XGB_PARAMS = dict(n_estimators=1000, learning_rate=0.02, max_depth=6,
                  subsample=0.8, colsample_bytree=0.8, min_child_weight=4,
                  random_state=RANDOM_STATE)

X_train, X_test, y_train, y_test = train_test_split(
    X_all.values.astype(float), y_all, test_size=0.15, random_state=RANDOM_STATE)

model = XGBRegressor(**XGB_PARAMS).fit(X_train, y_train)
mae = mean_absolute_error(np.expm1(y_test), np.expm1(model.predict(X_test)))
print(f"XGBoost MAE (original scale): {mae:.2f}  "
      f"(mean daily attendance: {model_df['Entries'].mean():.0f})")


## Flight feature impact check
Identical rows, hyperparameters, and split seed — one model with the 18 flight
features, one without — so the MAE delta is attributable to the flight data alone.

In [ ]:
import matplotlib.pyplot as plt

_flight_cols = [c for c in FLIGHT_FEATURE_COLS if c in X_all.columns]
_X_without = X_all.drop(columns=_flight_cols)

_impact = {}
for _label, _X in [("WITHOUT flight features", _X_without),
                   ("WITH flight features", X_all)]:
    _Xtr, _Xte, _ytr, _yte = train_test_split(
        _X.values.astype(float), y_all, test_size=0.15, random_state=RANDOM_STATE)
    _mdl = XGBRegressor(**XGB_PARAMS).fit(_Xtr, _ytr)
    _mae = mean_absolute_error(np.expm1(_yte), np.expm1(_mdl.predict(_Xte)))
    _impact[_label] = (_mae, _mdl, _X.columns)
    print(f"{_label:<26s} MAE = {_mae:.2f}")

_delta = _impact["WITHOUT flight features"][0] - _impact["WITH flight features"][0]
print(f"\nMAE change from adding flight features: {_delta:+.2f} "
      f"({'improvement' if _delta > 0 else 'regression'})")

_mae_w, _mdl_w, _cols_w = _impact["WITH flight features"]
_imp_df = (pd.DataFrame({"Feature": _cols_w, "Importance": _mdl_w.feature_importances_})
           .sort_values("Importance", ascending=False).reset_index(drop=True))
_fl = _imp_df[_imp_df["Feature"].isin(_flight_cols)]
print(f"\nFlight feature importances (rank of {len(_imp_df)} features):")
for _r, _row in _fl.iterrows():
    print(f"  #{_r + 1:>3d}  {_row['Feature']:<16s} {_row['Importance']:.4f}")

_top = _imp_df.head(20)
plt.figure(figsize=(10, 6))
plt.barh(_top["Feature"][::-1], _top["Importance"][::-1],
         color=["tomato" if f in _flight_cols else "steelblue" for f in _top["Feature"]][::-1])
plt.title("Top 20 feature importances (flight features in red)")
plt.xlabel("Relative importance")
plt.tight_layout(); plt.show()


## Chronological hold-out
First 80% of operating days to train, last 20% to test — the honest view of how
the model extrapolates forward in time.

In [ ]:
split = int(len(model_df) * 0.8)
Xc, yc = X_all.values.astype(float), y_all
Xc_tr, Xc_te, yc_tr, yc_te = Xc[:split], Xc[split:], yc[:split], yc[split:]

chrono = XGBRegressor(**XGB_PARAMS).fit(Xc_tr, yc_tr)
yc_pred = chrono.predict(Xc_te)
chrono_mae = mean_absolute_error(np.expm1(yc_te), np.expm1(yc_pred))
print(f"Chronological split MAE (original scale): {chrono_mae:.2f} "
      f"({len(Xc_te)} test days)")

_dates_te = df["Date"].iloc[split:]
plt.figure(figsize=(14, 5))
plt.plot(_dates_te, np.expm1(yc_te), label="Actual", marker="o", ms=3, lw=0.8)
plt.plot(_dates_te, np.expm1(yc_pred), label="Predicted", marker="x", ms=3,
         lw=0.8, ls="--")
plt.title(f"Chronological hold-out — actual vs predicted (MAE {chrono_mae:.0f})")
plt.legend(); plt.grid(alpha=0.4); plt.tight_layout(); plt.show()


## Notes & next steps

- **COVID confound.** 2020–21 removes both flights and attendance; if flight features
  dominate importances, re-run with `DROP_COVID = True` to see how much survives on
  normal years. A model intended for deployment should probably train post-reopening.
- **Switch parks** by changing `PARK` to `"Tivoli Gardens"` — airports and holiday
  calendar follow automatically. Note the bundled weather file corresponds to one
  location; check it matches the park you model.
- **Wait-time angle.** The dataset's 15-min `waiting_times` file supports the original
  Kaggle framing (predict queue length); daily attendance predicted here is a natural
  upstream feature for that model.
- **Deployment.** For live scoring, the `Curr_*`/`Next_*` windows would come from
  published airline schedules rather than EUROCONTROL actuals (which lag a few days
  to weeks). The AeroAPI `/schedules/{start}/{end}` endpoint with
  `destination=<IATA>` covers that — same pattern as the appendix in the previous
  notebook, using BCN/REU or CPH.
- The other experiments from the original (Prophet, LSTM, Ridge, target encoding)
  port over directly: everything they need is in `model_df`.

---
# Part 2 — Squeezing MAE

Levers in descending order of expected payoff, each measured properly before moving on:

1. **Measurement first.** A single 15% split can't detect a 5% gain at this sample size.
   Everything below is scored with 5-fold CV (random view) *and* expanding-window
   time-series CV (forecasting view), on a **dev set only** — the final 15% of days is
   locked away untouched until the last cell, so the tuning can't overfit the number
   we report.
2. **Objective alignment.** We report MAE but currently train squared error on
   `log1p`. Trying {raw, log1p} × {squared error, absolute error} costs 20 fits and
   often buys a few percent by itself.
3. **Features** — usually the biggest lever here: park **operating hours** and
   **show schedules** (known in advance → legitimate), **lagged wait times** (yesterday's
   crowding), richer holiday structure (Easter week, bridge days, distance-to-holiday),
   longer attendance lags (14, 364), same-weekday rolling means, and flight *ratios*
   that dampen the COVID level-shift.
4. **Hyperparameter search** (Optuna) — typically 5–15% over decent hand-set params.
5. **Seed-bagging + LightGBM blend** — a final 2–5%.

Leakage rules enforced throughout: nothing same-day that is a *consequence* of
attendance (wait times enter only lagged), and no target encodings fit outside CV
folds.

In [ ]:
%pip install -q optuna lightgbm


In [ ]:
# ---------- Evaluation harness + dev/holdout protocol ----------
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error

HOLDOUT_FRAC = 0.15                       # final days, untouched until the last cell
n_hold = int(len(model_df) * HOLDOUT_FRAC)
DEV = np.arange(len(model_df) - n_hold)   # rows are chronological
HOLD = np.arange(len(model_df) - n_hold, len(model_df))
print(f"Dev: {len(DEV)} days | Holdout: {len(HOLD)} days "
      f"({df['Date'].iloc[HOLD[0]]:%Y-%m-%d} -> {df['Date'].iloc[-1]:%Y-%m-%d})")

def wape(y_true, y_pred):
    return np.abs(y_true - y_pred).sum() / np.abs(y_true).sum()

def _fit_predict(params, Xtr, ytr_raw, Xte, transform):
    ytr = np.log1p(ytr_raw) if transform == "log1p" else ytr_raw
    m = XGBRegressor(**params).fit(Xtr, ytr)
    p = m.predict(Xte)
    return np.clip(np.expm1(p) if transform == "log1p" else p, 0, None)

def cv_mae(X, y_raw, params, transform="log1p", view="random", n_splits=5):
    """Mean/std MAE across folds. view='random' -> shuffled KFold;
    view='chrono' -> expanding-window TimeSeriesSplit (rows must be time-ordered)."""
    Xv = np.asarray(X, dtype=float)
    splitter = (KFold(n_splits, shuffle=True, random_state=RANDOM_STATE)
                if view == "random" else TimeSeriesSplit(n_splits))
    maes = []
    for tr, te in splitter.split(Xv):
        p = _fit_predict(params, Xv[tr], y_raw[tr], Xv[te], transform)
        maes.append(mean_absolute_error(y_raw[te], p))
    return float(np.mean(maes)), float(np.std(maes))

y_raw_all = model_df["Entries"].values.astype(float)
X_dev, y_dev = X_all.iloc[DEV], y_raw_all[DEV]

scoreboard = {}
def record(name, X, params, transform):
    r = cv_mae(X, y_dev, params, transform, "random")
    c = cv_mae(X, y_dev, params, transform, "chrono")
    scoreboard[name] = {"random": r, "chrono": c, "params": params, "transform": transform}
    print(f"{name:<28s} random {r[0]:7.1f} ±{r[1]:5.1f}   chrono {c[0]:7.1f} ±{c[1]:5.1f}")

print("\nBaseline (Part 1 params, log1p target):")
record("baseline", X_dev, XGB_PARAMS, "log1p")
print(f"(mean daily attendance {y_dev.mean():.0f} -> baseline random-CV WAPE "
      f"~{scoreboard['baseline']['random'][0] / y_dev.mean():.1%})")


## Lever 1 — target transform × training objective
`log1p` + squared error optimizes relative error; plain absolute error optimizes MAE
directly. Which wins depends on how heavy the attendance tail is — so we measure all
four. (`reg:absoluteerror` needs XGBoost ≥ 1.7; combos that fail are reported and
skipped.)

In [ ]:
grid_results = {}
for transform in ("log1p", "raw"):
    for obj in ("reg:squarederror", "reg:absoluteerror"):
        name = f"{transform} + {obj.split(':')[1]}"
        try:
            params = {**XGB_PARAMS, "objective": obj}
            r = cv_mae(X_dev, y_dev, params, transform, "random")
            grid_results[name] = (r, params, transform)
            print(f"{name:<28s} random-CV MAE {r[0]:7.1f} ±{r[1]:5.1f}")
        except Exception as e:
            print(f"{name:<28s} failed ({type(e).__name__}) — skipped")

best_name = min(grid_results, key=lambda k: grid_results[k][0][0])
(_, BASE_PARAMS, BEST_TRANSFORM) = grid_results[best_name]
print(f"\nBest combo: {best_name}")
record("best objective", X_dev, BASE_PARAMS, BEST_TRANSFORM)


## Lever 2 — Feature Pack 2

**Calendar structure:** Easter week (movable feast — huge for Spanish parks), the
Christmas–Reyes stretch, bridge days (workday squeezed between a holiday and a
weekend), and distance to the nearest holiday (capped at 30 days).

**Attendance dynamics:** lag 14, lag 364 (same season last year — NaN early on;
XGBoost handles missing natively so we don't drop rows), mean of the last 4 same
weekdays, 7-day rolling std, week-over-week diff.

**Flight ratios:** momentum (next week ÷ last week) and current level vs the trailing
month's weekly rate — these are level-free, so they carry signal *through* the COVID
regime change instead of just encoding it.

**Dataset extras (defensive — each prints added/skipped):** daily **operating hours**
(from the schedule file if usable, else the observed span of wait-time records) and
**lagged wait-time stats** (yesterday's / last week's mean posted wait — crowding
persistence, never same-day).

In [ ]:
from dateutil.easter import easter

feat = df.copy()          # df still has Date + all Part-1 features + Entries

country, subdiv = PARK_HOLIDAYS[PARK]
years = range(feat["Date"].dt.year.min() - 1, feat["Date"].dt.year.max() + 2)
hol2 = holidays_lib.country_holidays(country, subdiv=subdiv, years=years)

all_days = pd.date_range(feat["Date"].min() - pd.Timedelta(days=40),
                         feat["Date"].max() + pd.Timedelta(days=40), freq="D")
hol_dates = pd.DatetimeIndex([d for d in all_days if d in hol2])

def _holiday_distance(dates):
    idx = hol_dates.values
    pos = np.searchsorted(idx, dates.values)
    nxt = idx[np.clip(pos, 0, len(idx) - 1)]
    prv = idx[np.clip(pos - 1, 0, len(idx) - 1)]
    to_next = (nxt - dates.values).astype("timedelta64[D]").astype(int)
    since = (dates.values - prv).astype("timedelta64[D]").astype(int)
    return np.clip(to_next, 0, 30), np.clip(since, 0, 30)

feat["Days_To_Holiday"], feat["Days_Since_Holiday"] = _holiday_distance(feat["Date"])

easters = {y: pd.Timestamp(easter(y)) for y in years}
def _easter_flag(d):
    e = easters[d.year]
    return int(e - pd.Timedelta(days=7) <= d <= e + pd.Timedelta(days=1))
feat["Is_Easter_Week"] = feat["Date"].apply(_easter_flag)
feat["Is_Xmas_Period"] = (((feat["Date"].dt.month == 12) & (feat["Date"].dt.day >= 20)) |
                          ((feat["Date"].dt.month == 1) & (feat["Date"].dt.day <= 6))).astype(int)
feat["Is_Summer_Peak"] = feat["Date"].dt.month.isin([7, 8]).astype(int)

_next_is_hol = feat["Date"].apply(lambda d: (d + pd.Timedelta(days=1)) in hol2)
_prev_is_hol = feat["Date"].apply(lambda d: (d - pd.Timedelta(days=1)) in hol2)
feat["Is_Bridge_Day"] = (((feat["DayOfWeek"] == 0) & _next_is_hol) |
                         ((feat["DayOfWeek"] == 4) & _prev_is_hol)).astype(int) \
                        * (1 - feat["Is_Holiday"])

# --- attendance dynamics (operating-day lags; NaNs kept for XGBoost) ---
feat = feat.sort_values("Date").reset_index(drop=True)
feat["Entries_Lag_14"] = feat["Entries"].shift(14)
feat["Entries_Lag_364"] = feat["Entries"].shift(364)
feat["Entries_SameDOW_Mean4"] = (feat.groupby("DayOfWeek")["Entries"]
                                 .transform(lambda s: s.shift(1).rolling(4, min_periods=2).mean()))
feat["Entries_Roll_Std_7"] = feat["Entries"].shift(1).rolling(7).std()
feat["Entries_WoW_Diff"] = feat["Entries_Lag_1"] - feat["Entries_Lag_7"]

# --- level-free flight ratios ---
_eps = 1e-6
feat["Arr_Week_Momentum"] = feat["Arr_Next_Week"] / (feat["Arr_Last_Week"] + _eps)
feat["Dep_Week_Momentum"] = feat["Dep_Next_Week"] / (feat["Dep_Last_Week"] + _eps)
feat["Arr_Curr_vs_Trail"] = feat["Arr_Curr_Week"] / (feat["Arr_Last_Month"] * 7 / 30 + _eps)
feat["Dep_Curr_vs_Trail"] = feat["Dep_Curr_Week"] / (feat["Dep_Last_Month"] * 7 / 30 + _eps)
for c in ["Arr_Week_Momentum", "Dep_Week_Momentum", "Arr_Curr_vs_Trail", "Dep_Curr_vs_Trail"]:
    feat[c] = feat[c].replace([np.inf, -np.inf], np.nan).clip(0, 10)

# --- dataset extras: operating hours + lagged wait stats (best effort) ---
def _daily_time_span(fp, date_cands, start_cands, end_cands):
    head = pd.read_csv(fp, nrows=0, sep=None, engine="python")
    lowc = {c.lower(): c for c in head.columns}
    dcol = next((lowc[c] for c in date_cands if c in lowc), None)
    scol = next((lowc[c] for c in start_cands if c in lowc), None)
    ecol = next((lowc[c] for c in end_cands if c in lowc), None)
    if not (dcol and scol and ecol):
        raise ValueError(f"columns not recognized in {fp.name}: {list(head.columns)}")
    t = pd.read_csv(fp, usecols=list(dict.fromkeys([dcol, scol, ecol])), sep=None, engine="python")
    t[dcol] = pd.to_datetime(t[dcol], errors="coerce")
    for c in (scol, ecol):
        t[c] = pd.to_datetime(t[c], errors="coerce")
    t = t.dropna()
    g = t.groupby(t[dcol].dt.normalize()).agg(_open=(scol, "min"), _close=(ecol, "max"))
    hrs = ((g["_close"] - g["_open"]).dt.total_seconds() / 3600).clip(2, 24)
    return hrs.rename("Open_Hours").reset_index().rename(columns={dcol: "Date"})

added, skipped = [], []
sched_file = next((p for p in files if "schedule" in p.name.lower() and p.suffix.lower() == ".csv"), None)
wait_file = next((p for p in files if "waiting" in p.name.lower() and p.suffix.lower() == ".csv"), None)

hours_df = None
for src_file, label in [(sched_file, "entity schedule"), (wait_file, "wait-time span")]:
    if src_file is None or hours_df is not None:
        continue
    try:
        hours_df = _daily_time_span(src_file,
                                    ("work_date", "usage_date", "date", "deb_time"),
                                    ("deb_time", "open_time", "opening_time"),
                                    ("fin_time", "close_time", "closing_time"))
        added.append(f"Open_Hours (from {label})")
    except Exception as e:
        skipped.append(f"Open_Hours via {label}: {e}")
if hours_df is not None:
    feat = feat.merge(hours_df, on="Date", how="left")

try:
    if wait_file is None:
        raise ValueError("no waiting_times file found")
    head = pd.read_csv(wait_file, nrows=0, sep=None, engine="python")
    lowc = {c.lower(): c for c in head.columns}
    dcol = next((lowc[c] for c in ("work_date", "usage_date", "date") if c in lowc), None)
    wcol = next((lowc[c] for c in ("wait_time_max", "wait_time", "posted_wait") if c in lowc), None)
    if not (dcol and wcol):
        raise ValueError(f"columns not recognized: {list(head.columns)}")
    wt = pd.read_csv(wait_file, usecols=[dcol, wcol], sep=None, engine="python")
    wt[dcol] = pd.to_datetime(wt[dcol], errors="coerce")
    wt[wcol] = pd.to_numeric(wt[wcol], errors="coerce")
    daily_wait = (wt.dropna().groupby(wt[dcol].dt.normalize())[wcol]
                    .mean().rename("Wait_Mean"))
    cont = daily_wait.reindex(pd.date_range(daily_wait.index.min(), daily_wait.index.max()))
    lagged = pd.DataFrame({"Date": cont.index,
                           "Wait_Mean_Lag_1": cont.shift(1).values,
                           "Wait_Mean_Lag_7": cont.shift(7).values})
    feat = feat.merge(lagged, on="Date", how="left")
    added.append("Wait_Mean_Lag_1 / Wait_Mean_Lag_7 (calendar-day lags, never same-day)")
except Exception as e:
    skipped.append(f"lagged wait stats: {e}")

for msg in added:
    print("added  :", msg)
for msg in skipped:
    print("skipped:", msg)

# --- assemble Feature Pack 2 matrix (aligned row-for-row with model_df) ---
feat["Date_Num"] = feat["Date"].dt.strftime("%Y%m%d").astype(int)
X2_full = feat.drop(columns=["Date", "Entries"])
X2_full = X2_full.dropna(axis=1, how="all")          # drop extras that never populated
NEW_COLS = [c for c in X2_full.columns if c not in X_all.columns]
print(f"\n{len(NEW_COLS)} new features: {NEW_COLS}")
X2_dev = X2_full.iloc[DEV]

record("+ feature pack 2", X2_dev, BASE_PARAMS, BEST_TRANSFORM)


## Lever 3 — Optuna hyperparameter search
TPE over depth, learning rate, tree count, sampling, and both regularization terms,
scored by random-view CV on the dev set. `N_TRIALS = 60` takes roughly 10–20 min in
Colab at this data size; raise it if you're leaving it to run. If Optuna isn't
installed the cell falls back to the current params so the rest of the notebook still
runs.

In [ ]:
N_TRIALS = 60
TUNED_PARAMS = dict(BASE_PARAMS)

try:
    import optuna
    from optuna.samplers import TPESampler
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 2500, log=True),
            "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 9),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 20.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "objective": BASE_PARAMS.get("objective", "reg:squarederror"),
            "tree_method": "hist", "n_jobs": -1, "random_state": RANDOM_STATE,
        }
        mae, _ = cv_mae(X2_dev, y_dev, params, BEST_TRANSFORM, "random", n_splits=4)
        return mae

    study = optuna.create_study(direction="minimize",
                                sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
    TUNED_PARAMS.update(study.best_params)
    TUNED_PARAMS.update({"tree_method": "hist", "n_jobs": -1, "random_state": RANDOM_STATE})
    print(f"Best trial CV MAE: {study.best_value:.1f}")
    print("Best params:", study.best_params)
except ImportError:
    print("optuna not installed (`%pip install optuna`) — keeping current params.")

record("+ tuned", X2_dev, TUNED_PARAMS, BEST_TRANSFORM)


## Lever 4 — seed bagging + LightGBM blend
Five XGBoost seeds averaged (kills tree-building variance), optionally blended
50/50 with a LightGBM mirror of the tuned params. Different tree implementations
make usefully decorrelated errors.

In [ ]:
try:
    from lightgbm import LGBMRegressor
    _HAVE_LGBM = True
except ImportError:
    _HAVE_LGBM = False
    print("lightgbm not installed — seed-bagged XGBoost only.")

def _make_models():
    models = [("xgb", lambda seed: XGBRegressor(**{**TUNED_PARAMS, "random_state": seed}))]
    if _HAVE_LGBM:
        lgb_params = dict(
            n_estimators=TUNED_PARAMS.get("n_estimators", 1000),
            learning_rate=TUNED_PARAMS.get("learning_rate", 0.02),
            num_leaves=2 ** TUNED_PARAMS.get("max_depth", 6) - 1,
            subsample=TUNED_PARAMS.get("subsample", 0.8),
            colsample_bytree=TUNED_PARAMS.get("colsample_bytree", 0.8),
            reg_alpha=TUNED_PARAMS.get("reg_alpha", 0.0),
            reg_lambda=TUNED_PARAMS.get("reg_lambda", 0.0),
            objective=("mae" if TUNED_PARAMS.get("objective", "").endswith("absoluteerror")
                       else "rmse"),
            verbose=-1,
        )
        models.append(("lgbm", lambda seed: LGBMRegressor(**{**lgb_params, "random_state": seed})))
    return models

ENSEMBLE_SEEDS = [0, 1, 2, 3, 4]

def ensemble_predict(Xtr, ytr_raw, Xte, transform):
    ytr = np.log1p(ytr_raw) if transform == "log1p" else ytr_raw
    preds = []
    for _name, factory in _make_models():
        for seed in ENSEMBLE_SEEDS:
            m = factory(seed).fit(Xtr, ytr)
            preds.append(m.predict(Xte))
    p = np.mean(preds, axis=0)
    return np.clip(np.expm1(p) if transform == "log1p" else p, 0, None)

def cv_mae_ensemble(X, y_raw, transform, view, n_splits=5):
    Xv = np.asarray(X, dtype=float)
    splitter = (KFold(n_splits, shuffle=True, random_state=RANDOM_STATE)
                if view == "random" else TimeSeriesSplit(n_splits))
    maes = [mean_absolute_error(y_raw[te],
                                ensemble_predict(Xv[tr], y_raw[tr], Xv[te], transform))
            for tr, te in splitter.split(Xv)]
    return float(np.mean(maes)), float(np.std(maes))

r = cv_mae_ensemble(X2_dev, y_dev, BEST_TRANSFORM, "random")
c = cv_mae_ensemble(X2_dev, y_dev, BEST_TRANSFORM, "chrono")
scoreboard["+ ensemble"] = {"random": r, "chrono": c,
                            "params": TUNED_PARAMS, "transform": BEST_TRANSFORM}
print(f"{'+ ensemble':<28s} random {r[0]:7.1f} ±{r[1]:5.1f}   chrono {c[0]:7.1f} ±{c[1]:5.1f}")


## Final verdict — untouched holdout
The last 15% of days, never seen by any evaluation above. This is the number to
believe: baseline (Part 1 features + params) vs the full squeeze, trained on all
dev days, scored on the holdout.

In [ ]:
print(f"{'stage':<28s} {'random CV':>16s} {'chrono CV':>16s}")
for name, row in scoreboard.items():
    print(f"{name:<28s} {row['random'][0]:9.1f} ±{row['random'][1]:4.1f}"
          f"  {row['chrono'][0]:9.1f} ±{row['chrono'][1]:4.1f}")

Xb = np.asarray(X_all, float)
X2v = np.asarray(X2_full, float)
y_hold = y_raw_all[HOLD]

p_base = _fit_predict(XGB_PARAMS, Xb[DEV], y_raw_all[DEV], Xb[HOLD], "log1p")
p_full = ensemble_predict(X2v[DEV], y_raw_all[DEV], X2v[HOLD], BEST_TRANSFORM)

mae_base = mean_absolute_error(y_hold, p_base)
mae_full = mean_absolute_error(y_hold, p_full)
print(f"\nHOLDOUT ({len(HOLD)} final days)")
print(f"  Part 1 baseline : MAE {mae_base:8.1f}   WAPE {wape(y_hold, p_base):6.1%}")
print(f"  Full squeeze    : MAE {mae_full:8.1f}   WAPE {wape(y_hold, p_full):6.1%}")
print(f"  Improvement     : {mae_base - mae_full:+.1f}  "
      f"({(mae_base - mae_full) / mae_base:+.1%})")

import matplotlib.pyplot as plt
_dh = df["Date"].iloc[HOLD]
plt.figure(figsize=(14, 5))
plt.plot(_dh, y_hold, label="Actual", marker="o", ms=3, lw=0.8)
plt.plot(_dh, p_full, label="Full squeeze", marker="x", ms=3, lw=0.8, ls="--")
plt.plot(_dh, p_base, label="Baseline", ms=3, lw=0.8, ls=":", alpha=0.7)
plt.title(f"Holdout — baseline MAE {mae_base:.0f} vs squeezed {mae_full:.0f}")
plt.legend(); plt.grid(alpha=0.4); plt.tight_layout(); plt.show()


## Reading the results honestly, and what's left

- **Random vs chrono is the real story.** Random-CV MAE (with `Entries_Lag_1` in the
  matrix) answers "how well can we interpolate a missing day" — expect it to look
  flattering. Chrono/holdout answers "how well do we forecast forward" and will be
  meaningfully worse. Both are correct answers to different questions.
- **If your planning horizon is H ≥ 2 days**, features that need yesterday's actuals
  aren't available at prediction time. Re-run the scoreboard after
  `X2_dev = X2_dev.drop(columns=["Entries_Lag_1", "Entries_Roll_Mean_7",
  "Entries_Roll_Std_7", "Entries_WoW_Diff", "Entries_SameDOW_Mean4",
  "Wait_Mean_Lag_1"])` to see the honest H-day-ahead number.
- **The likely floor.** With operating hours, holiday structure, and lagged demand
  in the matrix, daily park-attendance models typically bottom out around
  ~8–15% WAPE chronologically; the remaining error is genuine day-level noise
  (weather shocks, group bookings, promos you can't see). If the squeezed WAPE is
  already in that band, more hyperparameter search is chasing seeds.
- **Highest-value additions beyond this notebook:** actual *school-vacation*
  calendars (Catalonia + the French zones that feed PortAventura, or Danish school
  breaks for Tivoli) — public-holiday calendars miss most of what fills a Spanish
  park in late June; ticket-price/promo calendars if reconstructable; and at
  deployment, remember weather enters as a *forecast*, so training on actual weather
  slightly flatters live performance.
- **COVID sensitivity still applies:** flip `DROP_COVID = True` and re-run Part 2 to
  confirm the gains aren't riding the reopening ramp.

---
# Supplement — "advance bookings": the leaky version vs the honest one

**Why not `Booked = U(0.20, 0.40) × Entries`?** A feature computed from the realized
target cannot exist at prediction time — on the morning you forecast day *D*, day *D*'s
attendance hasn't happened yet, so nothing derived from it is available. Any MAE gain
from such a feature is fabricated: the model just learns `Entries ≈ Booked / 0.3`.
It also fails as a *simulation* of real bookings: real advance sales are predictive
because of their systematic structure (booking curves, holiday/group skews, lead-time
dynamics), and multiplying the target by uniform noise destroys that structure while
injecting oracle knowledge in its place. The two cells below make this concrete —
**nothing in this section touches the real feature matrix, scoreboard, or holdout.**

The demo is worth keeping precisely because it shows what leakage looks like from the
inside: a suspicious MAE drop, one feature seizing the top of the importance chart, and
the "hardest" days suddenly becoming easy.

In [ ]:
# ================= LEAKAGE DEMO — deliberately wrong, fully quarantined =================
rng_demo = np.random.default_rng(RANDOM_STATE)
leak = rng_demo.uniform(0.20, 0.40, size=len(DEV)) * y_dev

X_leaky = X2_dev.copy()
X_leaky["Booked_Ahead_LEAKY"] = leak

res = {}
for name, Xd in [("honest features", X2_dev), ("+ LEAKY 'bookings'", X_leaky)]:
    res[name] = cv_mae(Xd, y_dev, TUNED_PARAMS, BEST_TRANSFORM, "random")
    print(f"{name:<22s} random-CV MAE {res[name][0]:7.1f} ±{res[name][1]:5.1f}")
_fake = res["honest features"][0] - res["+ LEAKY 'bookings'"][0]
print(f"-> apparent 'improvement': {_fake:+.1f} MAE — 100% fabricated "
      f"(the feature is 20-40% of the answer key)\n")

# Where the fabrication concentrates: the days the honest model finds hardest
from sklearn.model_selection import train_test_split
_idx_tr, _idx_te = train_test_split(np.arange(len(DEV)), test_size=0.2,
                                    random_state=RANDOM_STATE)
def _abs_errors(Xd):
    p = _fit_predict(TUNED_PARAMS, np.asarray(Xd, float)[_idx_tr], y_dev[_idx_tr],
                     np.asarray(Xd, float)[_idx_te], BEST_TRANSFORM)
    return np.abs(y_dev[_idx_te] - p)

e_honest, e_leaky = _abs_errors(X2_dev), _abs_errors(X_leaky)
worst = np.argsort(e_honest)[-15:]
print(f"15 hardest test days: honest mean error {e_honest[worst].mean():7.0f}"
      f" -> with leak {e_leaky[worst].mean():7.0f}"
      f"  ({1 - e_leaky[worst].mean() / max(e_honest[worst].mean(), 1e-9):.0%} 'better' — "
      "exactly the days a fake feature flatters most)")

# Importance takeover
_ytr = np.log1p(y_dev) if BEST_TRANSFORM == "log1p" else y_dev
_m = XGBRegressor(**TUNED_PARAMS).fit(np.asarray(X_leaky, float), _ytr)
_imp = (pd.DataFrame({"Feature": X_leaky.columns, "Importance": _m.feature_importances_})
        .sort_values("Importance", ascending=False).reset_index(drop=True))
_rank = _imp.index[_imp["Feature"] == "Booked_Ahead_LEAKY"][0] + 1
print(f"Booked_Ahead_LEAKY importance rank: #{_rank} of {len(_imp)} "
      f"(importance {_imp.loc[_rank - 1, 'Importance']:.3f})")

# Quarantine guarantee: the demo feature never reaches the real pipeline
assert "Booked_Ahead_LEAKY" not in X2_full.columns
assert not any("LEAKY" in c for c in X2_full.columns)
print("\nQuarantine check passed: leaky feature is NOT in the real feature matrix.")


## The honest path — as-of booking snapshots

Real advance bookings are leakage-safe when they enter as **as-of snapshots**:
`Booked_AsOf_Hd` = cumulative bookings for visit date *D* as they stood *H* days
before *D* (a value that genuinely existed on the prediction date — never the final
booked total). The merge function below implements exactly that contract, and it's
what a webstore/ticketing export (visit date, snapshot date, cumulative bookings)
drops straight into.

To prove the plumbing without faking signal, the placebo generator simulates a
booking system driven **only by advance-knowable drivers the model already has**
(seasonality, weekday, holidays) plus noise. Merged through the same function, it
should move MAE by roughly nothing — which is the point: a synthetic feature built
from information you already model is redundant, and one built from the target leaks.
There is no third option; new signal has to come from new *real* data.

In [ ]:
# ---------------- leakage-safe merge (use this with real data) ----------------
def merge_booking_snapshots(frame, bookings, horizons=(1, 7, 30)):
    """Add Booked_AsOf_{H}d columns to `frame` (must have a Date column).

    `bookings` is a long snapshot table:
        VISIT_DATE    - the day being visited
        SNAPSHOT_DATE - the day the count was observed
        CUM_BOOKED    - cumulative bookings for VISIT_DATE as of SNAPSHOT_DATE
    For each horizon H, uses the LAST snapshot taken on or before VISIT_DATE - H days,
    so every feature value existed on the prediction date."""
    b = bookings.copy()
    b["VISIT_DATE"] = pd.to_datetime(b["VISIT_DATE"])
    b["SNAPSHOT_DATE"] = pd.to_datetime(b["SNAPSHOT_DATE"])
    out = frame.copy()
    for H in horizons:
        elig = b[b["SNAPSHOT_DATE"] <= b["VISIT_DATE"] - pd.Timedelta(days=H)]
        snap = (elig.sort_values("SNAPSHOT_DATE")
                    .groupby("VISIT_DATE")["CUM_BOOKED"].last()
                    .rename(f"Booked_AsOf_{H}d").reset_index()
                    .rename(columns={"VISIT_DATE": "Date"}))
        out = out.merge(snap, on="Date", how="left")
    return out

# ------------- placebo generator: advance-knowable drivers ONLY -------------
def simulate_booking_snapshots(dates, seed=0):
    """Synthetic snapshot table driven by season/weekday/holiday + noise.
    Deliberately contains NO information from realized attendance."""
    r = np.random.default_rng(seed)
    dates = pd.to_datetime(pd.Series(dates)).drop_duplicates().sort_values()
    doy = dates.dt.dayofyear.values
    dow = dates.dt.dayofweek.values
    is_hol = dates.apply(lambda d: d in hol2).astype(int).values
    level = (3000 * (1 + 0.5 * np.sin(2 * np.pi * (doy - 100) / 365.25))
             * np.where(np.isin(dow, [5, 6]), 1.4, 1.0)
             * np.where(is_hol == 1, 1.3, 1.0)
             * np.exp(r.normal(0, 0.25, len(dates))))
    fill = {30: 0.35, 7: 0.70, 1: 0.95}          # booking-curve fractions
    rows = []
    for d, lv in zip(dates, level):
        for H, frac in fill.items():
            rows.append({"VISIT_DATE": d,
                         "SNAPSHOT_DATE": d - pd.Timedelta(days=H),
                         "CUM_BOOKED": lv * frac * np.exp(r.normal(0, 0.05))})
    return pd.DataFrame(rows)

placebo = simulate_booking_snapshots(feat["Date"], seed=RANDOM_STATE)
placebo_feats = merge_booking_snapshots(feat[["Date"]], placebo)
BOOKING_COLS = [c for c in placebo_feats.columns if c.startswith("Booked_AsOf_")]

X_placebo = X2_dev.copy()
for c in BOOKING_COLS:
    X_placebo[c] = placebo_feats[c].iloc[DEV].values

r_honest = cv_mae(X2_dev, y_dev, TUNED_PARAMS, BEST_TRANSFORM, "random")
r_placebo = cv_mae(X_placebo, y_dev, TUNED_PARAMS, BEST_TRANSFORM, "random")
r_leaky = res["+ LEAKY 'bookings'"]

print(f"{'honest features':<26s} random-CV MAE {r_honest[0]:7.1f} ±{r_honest[1]:5.1f}")
print(f"{'+ placebo bookings':<26s} random-CV MAE {r_placebo[0]:7.1f} ±{r_placebo[1]:5.1f}"
      "   <- ~no change: redundant with existing features, as it must be")
print(f"{'+ LEAKY bookings':<26s} random-CV MAE {r_leaky[0]:7.1f} ±{r_leaky[1]:5.1f}"
      "   <- fabricated gain: derived from the target")

print("\nWhen real snapshot exports exist (visit date, snapshot date, cumulative "
      "booked),\ncall merge_booking_snapshots() with them and add the Booked_AsOf_* "
      "columns to the\nreal matrix — leakage-safe by construction, and likely one of "
      "the strongest\nfeatures this model can get.")


## Where the placebo bookings land among the real features

Two lenses, because they answer different questions:

- **Gain importance** — how much the trees *used* each feature. Redundant features can
  score well here: the placebo columns correlate with seasonality, so the model may
  split on them in place of the calendar features they duplicate.
- **Permutation ΔMAE** — shuffle one feature's values on held-out days and measure how
  much validation MAE rises. This is what the feature actually *contributes* to
  predictions. For a redundant feature the answer is ≈ 0 (the rest of the matrix
  covers for it). Note the precise reading: ≈ 0 means "adds nothing beyond the other
  features," not "contains nothing."

The third panel is the punchline: shuffle cost of the three placebo columns vs the
LEAKY feature (in its own model). Real booking data, when it arrives, should land
somewhere in between — visible positive permutation ΔMAE, without the absurd
dominance of the leak.

In [ ]:
if "X_placebo" not in globals() or "X_leaky" not in globals():
    raise RuntimeError("Run the bookings supplement cells above first.")

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

_tr_i, _va_i = train_test_split(np.arange(len(DEV)), test_size=0.2,
                                random_state=RANDOM_STATE)

def _fit_on(X_frame):
    ytr = np.log1p(y_dev[_tr_i]) if BEST_TRANSFORM == "log1p" else y_dev[_tr_i]
    return XGBRegressor(**TUNED_PARAMS).fit(np.asarray(X_frame, float)[_tr_i], ytr)

def _val_mae_arr(model, X_arr):
    p = model.predict(X_arr)
    p = np.expm1(p) if BEST_TRANSFORM == "log1p" else p
    return mean_absolute_error(y_dev[_va_i], np.clip(p, 0, None))

def shuffle_costs(model, X_frame, cols, n_repeats=5, seed=0):
    """Permutation importance as ΔMAE: how much held-out MAE rises when one
    feature's validation values are shuffled. Returns {col: mean ΔMAE}."""
    r = np.random.default_rng(seed)
    X_arr = np.asarray(X_frame, float)
    X_va = X_arr[_va_i]
    base = _val_mae_arr(model, X_va)
    idx = {c: j for j, c in enumerate(X_frame.columns)}
    out = {}
    for c in cols:
        ds = []
        for _ in range(n_repeats):
            Xs = X_va.copy()
            Xs[:, idx[c]] = r.permutation(Xs[:, idx[c]])
            ds.append(_val_mae_arr(model, Xs) - base)
        out[c] = float(np.mean(ds))
    return out, base

# --- placebo model: gain importance + permutation ΔMAE for every feature ---
model_placebo = _fit_on(X_placebo)
gain_imp = (pd.DataFrame({"Feature": X_placebo.columns,
                          "Gain": model_placebo.feature_importances_})
            .sort_values("Gain", ascending=False).reset_index(drop=True))
perm_deltas, base_mae_placebo = shuffle_costs(model_placebo, X_placebo,
                                              list(X_placebo.columns),
                                              n_repeats=5, seed=RANDOM_STATE)
perm_imp = (pd.DataFrame({"Feature": list(perm_deltas), "DeltaMAE": list(perm_deltas.values())})
            .sort_values("DeltaMAE", ascending=False).reset_index(drop=True))

# --- leaky model: shuffle cost of the leak alone, for contrast ---
model_leaky = _fit_on(X_leaky)
leak_costs, _ = shuffle_costs(model_leaky, X_leaky, ["Booked_Ahead_LEAKY"],
                              n_repeats=5, seed=RANDOM_STATE)
leak_shuffle_delta = leak_costs["Booked_Ahead_LEAKY"]

# --- summary table ---
print(f"Validation MAE (placebo model): {base_mae_placebo:.1f}\n")
print(f"{'feature':<18s} {'gain rank':>9s} {'gain':>8s} {'perm ΔMAE':>10s}")
_median_perm = perm_imp["DeltaMAE"].median()
for c in BOOKING_COLS:
    g_rank = int(gain_imp.index[gain_imp["Feature"] == c][0]) + 1
    g_val = float(gain_imp.loc[gain_imp["Feature"] == c, "Gain"].iloc[0])
    print(f"{c:<18s} {g_rank:>6d}/{len(gain_imp):<3d} {g_val:>8.4f} {perm_deltas[c]:>+10.1f}")
print(f"{'(median feature)':<18s} {'':>9s} {'':>8s} {_median_perm:>+10.1f}")
print(f"{'Booked_Ahead_LEAKY':<18s} {'(own model)':>9s} {'':>8s} {leak_shuffle_delta:>+10.1f}")

# --- three-panel visual ---
fig, axes = plt.subplots(1, 3, figsize=(17, 6),
                         gridspec_kw={"width_ratios": [1.15, 1.15, 0.8]})

_top_g = gain_imp.head(20)
axes[0].barh(_top_g["Feature"][::-1], _top_g["Gain"][::-1],
             color=["darkorange" if f in BOOKING_COLS else "steelblue"
                    for f in _top_g["Feature"]][::-1])
axes[0].set_title("Gain importance\n(what the trees used)")
axes[0].set_xlabel("Gain share")

_top_p = perm_imp.head(20)
axes[1].barh(_top_p["Feature"][::-1], _top_p["DeltaMAE"][::-1],
             color=["darkorange" if f in BOOKING_COLS else "steelblue"
                    for f in _top_p["Feature"]][::-1])
axes[1].set_title("Permutation ΔMAE, held-out days\n(what actually helps)")
axes[1].set_xlabel("MAE increase when shuffled")
axes[1].axvline(0, color="gray", lw=0.8)

_names = BOOKING_COLS + ["Booked_Ahead_LEAKY"]
_vals = [perm_deltas[c] for c in BOOKING_COLS] + [leak_shuffle_delta]
_cols = ["darkorange"] * len(BOOKING_COLS) + ["tomato"]
bars = axes[2].bar(range(len(_names)), _vals, color=_cols)
axes[2].set_xticks(range(len(_names)))
axes[2].set_xticklabels([n.replace("Booked_", "") for n in _names],
                        rotation=30, ha="right")
axes[2].set_title("Shuffle cost:\nplacebo vs leaky")
axes[2].set_ylabel("MAE increase when shuffled")
axes[2].axhline(0, color="gray", lw=0.8)
for b, v in zip(bars, _vals):
    axes[2].annotate(f"{v:+.0f}", (b.get_x() + b.get_width() / 2, v),
                     ha="center", va="bottom" if v >= 0 else "top", fontsize=9)

for spine_ax in axes:
    spine_ax.grid(alpha=0.3, axis="x" if spine_ax is not axes[2] else "y")
fig.suptitle("Placebo booking features vs the rest of the matrix "
             "(orange = placebo, red = leaky demo)", y=1.02)
plt.tight_layout()
plt.show()

print("\nReading: orange bars may pick up some gain share (they proxy seasonality the"
      "\nmodel already has), but their shuffle cost sits at ~zero — they contribute"
      "\nnothing beyond the existing features. The leak's shuffle cost towers over"
      "\neverything, which is the signature of a feature carrying the answer key."
      "\nReal Booked_AsOf_* data should land in between: clearly positive ΔMAE,"
      "\nnowhere near the leak.")
